In [36]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import sklearn as sk
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
from sklearn.datasets import make_classification
from sklearn.preprocessing import StandardScaler
from collections import Counter

print("finished")

finished


## <a id='toc1_1_'></a>[KNN 算法](#toc0_)

原理：KNN 算法是一种基于距离度量的分类算法，其基本思想是：如果一个样本在特征空间中的 k 个最邻近的样本中的大多数属于某一类，则该样本也属于这个类。KNN 算法的实现过程可以分为以下几个步骤：

1. 计算样本之间的距离：首先需要计算样本之间的距离，常用的距离计算方法有欧式距离、曼哈顿距离、切比雪夫距离等。
2. 确定 k 值：KNN 算法的关键参数 k，即邻近的样本的数量。一般来说，k 值的选择对 KNN 算法的精度和效率都有很大的影响。
3. 确定类别：KNN 算法根据 k 个邻近样本的类别，决定待分类样本的类别。
4. 实现 KNN 算法：KNN 算法的实现过程可以分为以下几个步骤：
   - 1）计算待分类样本与样本库中每个样本之间的距离。
   - 2）按照距离递增次序排序。
   - 3）选取与待分类样本距离最小的 k 个样本。
   - 4）确定待分类样本的类别。


In [ ]:
class KNN:
    def __init__(self, k=3):
        self.k = k  # 取的个数
        # 训练集和标签
        self.X_train = None
        self.y_train = None

    def fit(self, X, y):
        self.X_train = X
        self.y_train = y

    def getDist(self, x1, x2):
        # return np.sqrt(np.sum(x1 - x2))
        return np.sqrt(np.sum((x1 - x2) ** 2))

    # 计算距离,给定单个数据点，返回类别
    def predict_one(self, x):
        distances = []
        for i in range(len(self.X_train)):
            dist = self.getDist(x, self.X_train[i])
            # 存储距离和标签
            distances.append((dist, self.y_train[i]))
            # 排序
        distances.sort(key=lambda x: x[0])
        # 取前k个
        k_neighbors = distances[:self.k]
        # 统计出现频率最高的标签
        # labels = max(k_neighbors, key=lambda x: x[1])[1]
        # 第二种实现方法
        labels = [label for _, label in k_neighbors]
        labels = Counter(labels).most_common(1)[0][0]
        return labels

    def predict(self, X):
        y_pred = []
        for x in X:
            y_pred.append(self.predict_one(x))
        return y_pred

    def score(self, X, y):
        y_pred = self.predict(X)
        return np.sum(y_pred == y) / len(y)

In [38]:
def test_KNN():
    knn = KNN()
    X_train = np.array([[1, 2], [2, 3], [3, 1], [4, 3], [5, 2],
                        [6, 4], [7, 5], [8, 6], [9, 7], [10, 8]]
                       )
    y_train = np.array([0, 0, 0, 1, 1, 1, 1, 1, 1, 1])
    X_test = np.array([[1, 1], [6, 5], [7, 6], [8, 7], [9, 8]])
    # knn.fit(X_train, y_train)
    knn.fit(X_train, y_train)
    y_pred = knn.predict(X_test)
    print(y_pred)


test_KNN()

[0, 1, 1, 1, 1]


## <a id='toc1_2_'></a>[逻辑回归模型](#toc0_)

逻辑回归模型是一种分类模型，它可以用来预测某个变量的取值为 0 或 1，或者是某一事件发生的概率。

逻辑回归模型的假设是：输入变量 X 与输出变量 Y 之间存在一个线性关系，即：

$$Y = \beta_0 + \beta_1X$$

其中，$\beta_0$和$\beta_1$是模型的参数，分别表示截距和斜率。

逻辑回归模型的损失函数为：

$$L(\beta_0, \beta_1) = -\frac{1}{n}\sum_{i=1}^n[y_i\log(\hat{y_i}) + (1-y_i)\log(1-\hat{y_i})]$$


In [ ]:
class LogisticRegression:
    def __init__(self, learning_rate=0.01, n_iters=1000):
        self.learning_rate = learning_rate
        self.n_iters = n_iters
        self.weights = None
        self.bias = None

    # 初始化参数
    def init_Params(self, n_features):
        self.weights = np.random.rand(n_features) * 0.01
        self.bias = 0.1

    def sigmoid(self, z):
        # z = wx + b
        return 1 / (1 + np.exp(-z))

    def compute_loss(self, y, y_pred):
        # 计算损失函数, 交叉熵损失函数
        # 参数是真实值和预测值
        epsilon = 1e-15
        # 防止log(0)
        y_pred = np.clip(y_pred, epsilon, 1 - epsilon)
        loss = -np.mean(y * np.log(y_pred) + (1 - y) * np.log(1 - y_pred))
        return loss

    def fit(self, X, y):
        '''
        
        :param X: (n_samples,n_features) 输入数据 ， 
        :param y: (n_samples,1) 预测值 
        :return: self 训练好的模型 
        '''
        n_samples, n_features, = X.shape
        self.init_Params(n_features)
        # 训练步骤：计算预测值，计算损失函数，更新参数
        for i in range(self.n_iters):
            # 计算预测值
            z = np.dot(X, self.weights) + self.bias
            y_pred = self.sigmoid(z)
            # 计算损失函数
            loss = self.compute_loss(y, y_pred)
            # 梯度计算公式:
            # ∂L/∂w = (1/n) * X^T · (y_pred - y)
            # ∂L/∂b = (1/n) * Σ (y_pred - y)
            dw = (1 / n_samples) * np.dot(X.T, (y_pred - y))
            db = (1 / n_samples) * np.sum(y_pred - y)
            # 更新参数
            self.weights -= self.learning_rate * dw
            self.bias -= self.learning_rate * db
            # 打印损失函数
            if i % 100 == 0:
                print(f"Iteration: {i}, Loss: {loss}")

        return self

    # 预测属于某一类的概率，也就是计算预测值，按值分类
    def predict_proba(self, X):
        z = np.dot(X, self.weights) + self.bias
        y_pred = self.sigmoid(z)
        return y_pred

    # 预测属于某一类的标签
    def predict(self, X, threshold=0.5):
        '''
        假设y_pred_proba=[0.2, 0.6, 0.7]，阈值 threshold=0.5：
        条件判断结果：[False, True, True]
        最终输出：y_pred = [0, 1, 1]
        '''
        y_pred_proba = self.predict_proba(X)
        y_pred = np.where(y_pred_proba > threshold, 1, 0)
        return y_pred

    # 计算准确率
    def score(self, X, y):
        y_pred = self.predict(X)
        accuracy = np.mean(y_pred == y)
        # y_pred是[1,0,1]，而y是[1,1,0]，那么y_pred == y就会得到[True, False, False], 此时平均值就是准确率
        return accuracy




1. 生成模拟数据集...
数据集形状: X=(1000, 4), y=(1000,)
2. 数据预处理...
数据集形状: X_scaled=(1000, 4)
3. 划分训练集和测试集...
训练集形状: X_train=(800, 4), y_train=(800,)
测试集形状: X_test=(200, 4), y_test=(200,)
4. 训练模型...
Iteration: 0, Loss: 0.6918796784739653
Iteration: 100, Loss: 0.5685435661183956
Iteration: 200, Loss: 0.49196013226617497
Iteration: 300, Loss: 0.4415365409838803
Iteration: 400, Loss: 0.40637740918857546
Iteration: 500, Loss: 0.38066516243633314
Iteration: 600, Loss: 0.3611301057201564
Iteration: 700, Loss: 0.3458277997258016
Iteration: 800, Loss: 0.3335410007778967
Iteration: 900, Loss: 0.3234731744196385
模型训练完成...
5. 评估模型性能...
训练集准确率: 0.90
测试集准确率: 0.88
6. 查看模型参数...
模型权重: [ 1.7668306  -0.05293393 -0.03600072 -0.07784187]
模型偏置: 0.031942691037757515
7. 预测...
样本标签: [0 1 0 1 0 1 0 1 1 1]
样本预测标签: [0 0 0 1 0 1 0 0 1 0]
样本预测概率: [0.07922162 0.49737992 0.10380977 0.83374862 0.11044406 0.50953297
 0.10266202 0.45545516 0.95365132 0.39753025]


## <a id='toc1_3_'></a>[测试逻辑回归模型](#toc0_)

我们可以用 make_classification()函数生成一些随机数据，然后用逻辑回归模型进行训练和预测。

make_classification()能生成一些带有噪声的分类数据，包括有标签的数据和无标签的数据。

StandardScaler()可以对数据进行标准化处理，使得每个特征的方差为 1，均值为 0。

scaler.fit_transform(X)可以对数据进行标准化处理, 并返回标准化后的数据。

标准化公式如下，其中$\mu$是均值，$\sigma$是方差：

$$x_i' = \frac{x_i - \mu}{\sigma}$$

train_test_split()可以将数据集划分为训练集和测试集,接受的参数有：

- X: 输入数据
- y: 输出数据
- test_size: 测试集占比
- random_state: 随机种子


In [ ]:
def test_LogisticRegression():
    # 生成数据:
    print("1. 生成模拟数据集...")

    X, y = make_classification(
        n_samples=1000,  # 总样本数
        n_features=4,  # 特征数量
        n_informative=2,  # 有信息的特征数,其他两个没啥用
        n_redundant=0,  # 冗余特征数
        n_clusters_per_class=1,
        random_state=42  # 随机种子
    )
    print(f"数据集形状: X={X.shape}, y={y.shape}")

    # 数据预处理
    print("2. 数据预处理...")
    scaler = StandardScaler()  # 得到标准化处理器
    X_scaled = scaler.fit_transform(X)  # 标准化处理
    # 先计算均值与方差，再拿计算结果来标准化数据
    print(f"数据集形状: X_scaled={X_scaled.shape}")

    #划分 train/test 数据集
    print("3. 划分训练集和测试集...")
    X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=42)
    print(f"训练集形状: X_train={X_train.shape}, y_train={y_train.shape}")
    print(f"测试集形状: X_test={X_test.shape}, y_test={y_test.shape}")

    # 训练模型
    print("4. 训练模型...")
    model = LogisticRegression()
    model.fit(X_train, y_train)
    print("模型训练完成...")

    # 评估性能
    print("5. 评估模型性能...")
    train_accuracy = model.score(X_train, y_train)
    test_accuracy = model.score(X_test, y_test)
    print(f"训练集准确率: {train_accuracy:.2f}")
    print(f"测试集准确率: {test_accuracy:.2f}")

    # 查看模型参数
    print("6. 查看模型参数...")
    print(f"模型权重: {model.weights}")
    print(f"模型偏置: {model.bias}")

    # 预测
    print("7. 预测...")
    sample_indices = range(10)
    X_samples = X_test[sample_indices]
    y_samples = y_test[sample_indices]
    y_pred = model.predict(X_samples)
    y_pred_proba = model.predict_proba(X_samples)
    print(f"样本标签: {y_samples}")
    print(f"样本预测标签: {y_pred}")
    print(f"样本预测概率: {y_pred_proba}")


test_LogisticRegression()



1. 生成模拟数据集...
数据集形状: X=(1000, 4), y=(1000,)
2. 数据预处理...
数据集形状: X_scaled=(1000, 4)
3. 划分训练集和测试集...
训练集形状: X_train=(800, 4), y_train=(800,)
测试集形状: X_test=(200, 4), y_test=(200,)
4. 训练模型...
Iteration: 0, Loss: 0.6930739633201418
Iteration: 100, Loss: 0.5692697409022943
Iteration: 200, Loss: 0.492426018604295
Iteration: 300, Loss: 0.4418542957658561
Iteration: 400, Loss: 0.40660582767616704
Iteration: 500, Loss: 0.38083646137554794
Iteration: 600, Loss: 0.36126298801930923
Iteration: 700, Loss: 0.34593372850799914
Iteration: 800, Loss: 0.3336273415368959
Iteration: 900, Loss: 0.32354485448665643
模型训练完成...
5. 评估模型性能...
训练集准确率: 0.90
测试集准确率: 0.88
6. 查看模型参数...
模型权重: [ 1.76616264 -0.05231203 -0.03647102 -0.07772502]
模型偏置: 0.03183626294127122
7. 预测...
样本标签: [0 1 0 1 0 1 0 1 1 1]
样本预测标签: [0 0 0 1 0 1 0 0 1 0]
样本预测概率: [0.07923943 0.49730887 0.10394314 0.83364642 0.11040969 0.50919867
 0.10282298 0.45559895 0.95351549 0.39770715]


## <a id='toc1_4_'></a>[Kmeans 算法](#toc0_)

Kmeans 算法是一种无监督学习算法，它可以用来对数据集进行聚类。

Kmeans 算法的基本思想是：

- 1）随机选择 k 个初始质心（centroids）
- 2）将数据集中的每个点分配到离它最近的质心
- 3）重新计算质心，使得每个质心所包含的点的均值更加接近真实的均值
- 4）重复步骤 2 和 3，直到质心不再移动


In [ ]:
#!/usr/bin/env python
# -*- coding: UTF-8 -*-
"""
@Project     ：MachineLearning
@File        ：K_means.py
@Description ：k—means聚类算法
@Author      ：Hello Worlds
@Date        ：2025/5/30 上午10:50
"""
import numpy as np
from sklearn.datasets import make_blobs
from sklearn.metrics import pairwise_distances, adjusted_rand_score, silhouette_score
import matplotlib.pyplot as plt

# 设置matplotlib支持中文显示
plt.rcParams['font.sans-serif'] = ['SimHei', 'Microsoft YaHei', 'DejaVu Sans']  # 设置中文字体
plt.rcParams['axes.unicode_minus'] = False  # 解决负号显示问题


class KMeans:
    def __init__(self,
                 k=3,
                 max_iter=100,
                 tol=1e-4,
                 random_state=0):
        """
        初始化K-Means算法

        参数:
        k: 聚类数量
        max_iter: 最大迭代次数
        tol: 收敛阈值（中心点移动距离）
        random_state: 随机种子
        """
        self.k = k
        self.max_iter = max_iter
        self.tol = tol
        self.random_state = random_state
        self.centroids = None  # 聚类中心
        self.labels = None  # 聚类标签
        self.inertia = None  # 聚类总平方误差(SSE)
        self.n_iter_ = 0  # 实际迭代次数

    def _initialize_centroids(self,
                              X):
        """
        初始化聚类中心,随机选择k个样本作为初始聚类中心,
        参数：n：样本数量，k：聚类中心数量，replace：是否可以重复选择
        return 初始聚类中心
        """
        np.random.seed(self.random_state)  # 设置随机种子
        # 随机选择k个样本作为初始聚类中心,
        random_indices = np.random.choice(X.shape[0], self.k, replace=False)
        # 选择的样本作为初始聚类中心
        return X[random_indices]

    def _assign_clusters(self,
                         X,
                         centroids):
        """
        计算每个样本到聚类中心的距离，并将样本分配到离它最近的聚类中心
        参数：X：样本数据，centroids：当前聚类中心
        return 样本所属的聚类标签
        """
        # 计算每个样本到聚类中心的距离
        distances = np.linalg.norm(X[:, np.newaxis, :] - centroids, axis=2)
        # 返回值：样本到各聚类中心的距离矩阵
        return np.argmin(distances, axis=1)

    def _update_centroids(self,
                          X,
                          labels):
        """
        更新聚类中心
        参数：X：样本数据，labels：样本所属的聚类标签
        return 新的聚类中心
        """
        # 初始化新的聚类中心
        new_centroids = np.zeros((self.k, X.shape[1]))
        for i in range(self.k):
            # 选择属于第i类的样本
            X_i = X[labels == i]
            if len(X_i) > 0:  # 防止聚类中心为全零
                # 更新第i类聚类中心,更新均值作为新的聚类中心
                new_centroids[i] = X_i.mean(axis=0)
            else:
                new_centroids[i] = X[np.random.randint(len(X))]  # 防止聚类中心为全零
        return new_centroids

    def _calculate_inertia(self,
                           X,
                           labels,
                           centroids):
        """
        计算聚类总平方误差(SSE)
        参数：X：样本数据，labels：样本所属的聚类标签，centroids：当前聚类中心
        return 聚类总平方误差(SSE)
        """
        inertia = 0  # 初始化聚类总平方误差(SSE)
        for i in range(self.k):
            # 选择属于第i类的样本
            X_i = X[labels == i]
            if len(X_i) > 0:  # 防止聚类中心为全零
                # 计算第i类样本到聚类中心的距离
                inertia += np.sum((X_i - centroids[i]) ** 2)
        return inertia

    def fit(self,
            X):
        """
        训练模型
        参数：X：样本数据（n_samples, n_features）
        return 训练好的模型
        """
        self.centroids = self._initialize_centroids(X)  # 初始化聚类中心
        for i in range(self.max_iter):
            self.n_iter_ = i + 1
            # 计算旧的聚类中心，用于判断收敛
            old_centroids = self.centroids.copy()
            # E-step: 将每个样本分配到离它最近的聚类中心
            self.labels = self._assign_clusters(X, self.centroids)
            # M-step: 更新聚类中心
            self.centroids = self._update_centroids(X, self.labels)
            # 计算误差平方和
            self.inertia = self._calculate_inertia(X, self.labels, self.centroids)
            # 判断收敛
            centroid_shift = np.linalg.norm(self.centroids - old_centroids)
            if centroid_shift < self.tol:
                print(f"已收敛，迭代次数：{i + 1}")
                break
        return self

    def predict(self,
                X):
        """
        预测样本所属的聚类
        参数：X：样本数据（n_samples, n_features）
        return 样本所属的聚类标签
        """
        if self.centroids is None:
            raise ValueError("模型没有训练,请先调用fit方法训练模型")
        return self._assign_clusters(X, self.centroids)

    def fit_predict(self, X):
        """
        训练模型并预测样本所属的聚类
        参数：X：样本数据（n_samples, n_features）
        return 样本所属的聚类标签
        """
        self.fit(X)
        return self.labels




In [ ]:
def test_KMeans():
    """测试手写KMeans算法并与sklearn版本比较"""
    # 生成数据
    print("生成测试数据...")
    X, y_true = make_blobs(n_samples=1000, centers=3, n_features=2,
                           cluster_std=0.8, random_state=42
                           )

    print("正在运行手写K-Means算法...")
    kmeans = KMeans(k=3, max_iter=100, tol=1e-4, random_state=42)
    labels = kmeans.fit_predict(X)
    centroids = kmeans.centroids

    # 使用sklearn的KMeans算法进行验证
    print("正在运行sklearn K-Means算法...")
    from sklearn.cluster import KMeans as SKKMeans
    sk_kmeans = SKKMeans(n_clusters=3, random_state=42, n_init=10)
    sk_labels = sk_kmeans.fit_predict(X)
    sk_centroids = sk_kmeans.cluster_centers_

    # 计算评估指标
    print("\n=== 算法性能评估 ===")

    # 1. Inertia比较
    print(f"手写K-Means的 inertia: {kmeans.inertia:.4f}")
    print(f"sklearn K-Means的 inertia: {sk_kmeans.inertia_:.4f}")
    print(f"Inertia差异: {abs(kmeans.inertia - sk_kmeans.inertia_):.6f}")

    # 2. 调整兰德指数 (ARI) - 与真实标签比较
    ari_handmade = adjusted_rand_score(y_true, labels)
    ari_sklearn = adjusted_rand_score(y_true, sk_labels)
    print(f"手写K-Means ARI: {ari_handmade:.4f}")
    print(f"sklearn K-Means ARI: {ari_sklearn:.4f}")

    # 3. 轮廓系数
    silhouette_handmade = silhouette_score(X, labels)
    silhouette_sklearn = silhouette_score(X, sk_labels)
    print(f"手写K-Means 轮廓系数: {silhouette_handmade:.4f}")
    print(f"sklearn K-Means 轮廓系数: {silhouette_sklearn:.4f}")

    # 4. 质心位置比较
    centroid_diff = np.linalg.norm(centroids - sk_centroids)
    print(f"质心位置差异 (Frobenius范数): {centroid_diff:.6f}")

    # 可视化结果
    plt.figure(figsize=(18, 6))

    # 子图1: 原始数据
    plt.subplot(1, 4, 1)
    plt.scatter(X[:, 0], X[:, 1], c=y_true, cmap='viridis', alpha=0.7, s=30)
    plt.title('原始数据 (真实标签)')
    plt.xlabel('特征1')
    plt.ylabel('特征2')
    plt.grid(True, alpha=0.3)

    # 子图2: 手写KMeans结果
    plt.subplot(1, 4, 2)
    plt.scatter(X[:, 0], X[:, 1], c=labels, cmap='viridis', alpha=0.7, s=30)
    plt.scatter(centroids[:, 0], centroids[:, 1], c='red', marker='X',
                s=200, label='质心', edgecolors='black'
                )
    plt.title('手写K-Means聚类结果')
    plt.xlabel('特征1')
    plt.ylabel('特征2')
    plt.legend()
    plt.grid(True, alpha=0.3)

    # 子图3: sklearn KMeans结果
    plt.subplot(1, 4, 3)
    plt.scatter(X[:, 0], X[:, 1], c=sk_labels, cmap='viridis', alpha=0.7, s=30)
    plt.scatter(sk_centroids[:, 0], sk_centroids[:, 1], c='red', marker='X',
                s=200, label='质心', edgecolors='black'
                )
    plt.title('sklearn K-Means聚类结果')
    plt.xlabel('特征1')
    plt.ylabel('特征2')
    plt.legend()
    plt.grid(True, alpha=0.3)

    # 子图4: 质心位置对比
    plt.subplot(1, 4, 4)
    plt.scatter(centroids[:, 0], centroids[:, 1], c='blue', marker='o',
                s=150, label='手写质心', alpha=0.8
                )
    plt.scatter(sk_centroids[:, 0], sk_centroids[:, 1], c='red', marker='s',
                s=150, label='sklearn质心', alpha=0.8
                )

    # 绘制连接线显示对应关系
    for i in range(len(centroids)):
        plt.plot([centroids[i, 0], sk_centroids[i, 0]],
                 [centroids[i, 1], sk_centroids[i, 1]],
                 'k--', alpha=0.5
                 )

    plt.title('质心位置对比')
    plt.xlabel('特征1')
    plt.ylabel('特征2')
    plt.legend()
    plt.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()

    return kmeans, sk_kmeans, X, y_true


def elbow_method(X, max_k=8):
    """肘部法则确定最佳K值"""
    print("\n=== 使用肘部法则确定最佳K值 ===")

    inertias = []
    k_values = range(1, max_k + 1)

    for k in k_values:
        kmeans = KMeans(k=k, random_state=42)
        kmeans.fit(X)
        inertias.append(kmeans.inertia)
        print(f"K={k}, inertia={kmeans.inertia:.2f}, 迭代次数={kmeans.n_iter_}")

    # 绘制肘部图
    plt.figure(figsize=(10, 6))
    plt.plot(k_values, inertias, 'bo-', linewidth=2, markersize=8)
    plt.xlabel('聚类数量 K')
    plt.ylabel('平方误差和 (Inertia)')
    plt.title('肘部法则 - 寻找最佳K值')
    plt.xticks(k_values)
    plt.grid(True, alpha=0.3)
    plt.show()

    return inertias


In [ ]:
def test_different_initializations():
    """测试不同随机初始化的稳定性"""
    print("\n=== 测试不同随机初始化的稳定性 ===")

    X, _ = make_blobs(n_samples=300, centers=3, random_state=42)

    inertias = []
    for i in range(10):
        kmeans = KMeans(k=3, random_state=i)  # 不同的随机种子
        kmeans.fit(X)
        inertias.append(kmeans.inertia)
        print(f"随机种子 {i}: inertia={kmeans.inertia:.2f}")

    print(f"\nInertia标准差: {np.std(inertias):.4f}")
    print(f"Inertia范围: {min(inertias):.2f} - {max(inertias):.2f}")


def run_Kmeans():
    print("开始测试手写K-Means聚类算法")
    print("=" * 50)

    # 测试主要功能
    kmeans_handmade, kmeans_sklearn, X, y_true = test_KMeans()

    # 使用肘部法则
    inertias = elbow_method(X, max_k=8)

    # 测试不同初始化的稳定性
    test_different_initializations()

    print("\n测试完成！")


run_Kmeans()

**Table of contents**<a id='toc0_'></a>    
- [KNN 算法](#toc1_1_)    
  - [逻辑回归模型](#toc1_2_)    
  - [测试逻辑回归模型](#toc1_3_)    
  - [Kmeans 算法](#toc1_4_)    
  - [随机森林和决策树](#toc1_5_)    
    - [随机森林核心参数](#toc1_5_1_)    
    - [随机森林特有参数](#toc1_5_2_)    
    - [决策树通用参数](#toc1_5_3_)    
    - [性能与计算参数](#toc1_5_4_)    
    - [回归树特有参数](#toc1_5_5_)    
    - [参数组合示例](#toc1_5_6_)    

<!-- vscode-jupyter-toc-config
	numbering=false
	anchor=true
	flat=false
	minLevel=1
	maxLevel=6
	/vscode-jupyter-toc-config -->
<!-- THIS CELL WILL BE REPLACED ON TOC UPDATE. DO NOT WRITE YOUR TEXT IN THIS CELL -->

## <a id='toc1_5_'></a>[随机森林和决策树](#toc0_)
### <a id='toc1_5_1_'></a>[随机森林核心参数](#toc0_)
- **`n_estimators=50`**  
  树的数量，控制模型复杂度与稳定性，增加树数可降低方差但提高计算成本。

- **`max_depth=5`**  
  树的最大深度，限制决策树生长，防止过拟合，深度过深易过拟合，过浅易欠拟合。

- **`min_samples_split=2`**  
  节点分裂所需最小样本数，低于该值的节点不再分裂，控制树生长防止过拟合。

- **`min_samples_leaf=1`**  
  叶子节点最少样本数，低于该值的叶子可能被剪枝，与分裂参数配合控制复杂度。

- **`random_state=42`**  
  随机数种子，固定训练中的随机性（如数据抽样、树生成顺序），确保结果可复现。

### <a id='toc1_5_2_'></a>[随机森林特有参数](#toc0_)
- **`bootstrap=True`**  
  是否启用有放回抽样，`False`时使用全量数据训练每棵树，降低多样性。

- **`max_features='sqrt'`**  
  分裂时考虑的最大特征数，常用选项：  
  - `'sqrt'`：总特征数的平方根（分类默认）  
  - `'log2'`：总特征数的对数（回归默认）  
  - `None`：使用所有特征（可能过拟合）

- **`oob_score=False`**  
  是否计算袋外样本准确率，`True`时输出未参与训练样本的评估分数。

- **`class_weight=None`**  
  处理类别不平衡，`'balanced'`自动调整权重，或自定义字典指定类别权重。

### <a id='toc1_5_3_'></a>[决策树通用参数](#toc0_)
- **`criterion='gini'`**  
  分裂标准：分类树用`'gini'`（基尼系数）或`'entropy'`（信息增益）；回归树用`'squared_error'`（均方误差）或`'absolute_error'`（平均绝对误差）。

- **`splitter='best'`**  
  分裂策略：`'best'`选择最优分裂点（计算成本高），`'random'`随机选择分裂点（加速训练）。

- **`ccp_alpha=0.0`**  
  代价复杂度剪枝参数，值越大剪枝越激进，需配合`cost_complexity_pruning_path`使用。

### <a id='toc1_5_4_'></a>[性能与计算参数](#toc0_)
- **`n_jobs=-1`**  
  并行计算线程数，`-1`表示使用所有CPU核心，加速训练（尤其大数据集）。

- **`warm_start=False`**  
  是否复用之前树结构增量训练，`True`时支持增量学习。

- **`verbose=0`**  
  训练日志详细程度，值越高输出信息越多（如训练进度）。

### <a id='toc1_5_5_'></a>[回归树特有参数](#toc0_)
- **`min_weight_fraction_leaf=0.0`**  
  叶子节点最小权重和，考虑样本权重时使用（如样本重要性不同）。

- **`max_leaf_nodes=None`**  
  叶子节点最大数量，与`max_depth`二选一控制树生长。

### <a id='toc1_5_6_'></a>[参数组合示例](#toc0_)
```python
from sklearn.ensemble import RandomForestClassifier

model = RandomForestClassifier(
    n_estimators=100,       # 树数量
    max_depth=10,           # 最大深度
    min_samples_split=5,    # 分裂最小样本数
    min_samples_leaf=2,     # 叶子最小样本数
    max_features='sqrt',    # 每棵树特征数
    bootstrap=True,         # 启用有放回抽样
    random_state=42,        # 随机种子
    n_jobs=-1,             # 并行计算
    class_weight='balanced' # 处理类别不平衡
)
model.fit(X_train, y_train)

In [2]:
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler

def use_RandomForest(train, test):
    """
    随机森林算法
    """
    train = np.array(train)
    test = np.array(test)
    # 分离特征和标签
    X_train, y_train = train[:, :-1], train[:, -1]
    # 标准化
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(test)
    # 处理完数据，该模型了
    model = RandomForestClassifier(
        n_estimators=50,
        # max_depth=5,
        random_state=42,
        min_samples_split=2,
        min_samples_leaf=1,
    )
    # 划分训练集和测试集
    model.fit(X_train_scaled, y_train)
    # 预测,astype(int)将结果转化为int型,否则是（1.,0.）
    y_pred = model.predict(X_test_scaled).astype(int)
    # 计算准确率
    return y_pred


def commit():
    train_example = [[1.0, 2.0, 0], [2.0, 3.0, 1], [3.0, 4.0, 0], [4.0, 5.0, 1]]

    test_example = [[1.5, 2.5], [3.5, 4.5]]

    # 进行预测
    result = use_RandomForest(train_example, test_example)
    print("预测结果:", result)


commit()

预测结果: [1 0]
